# Trabalho final

Gabriel Ferreira de Souza - gfs2@cesar.school
Arthur Avila - aoa2@cesar.school

Dataset escolhido:

Neural Networks Homer and Bart Classification (https://www.kaggle.com/datasets/juniorbueno/neural-networks-homer-and-bart-classification)

## Importação das bibliotecas

In [ ]:
# http://pytorch.org/
import torch

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import random_split

In [ ]:
import kagglehub
import os

# Baixa o dataset
path = kagglehub.dataset_download(
    "juniorbueno/neural-networks-homer-and-bart-classification"
)

base_path = os.path.join(path, "homer_bart_1")

print("Verificando conteudo:", os.listdir(base_path))

100%|██████████| 8.06M/8.06M [00:00<00:00, 99.8MB/s]

Extracting files...


Verificando conteudo: ['homer74.bmp', 'homer93.bmp', 'homer65.bmp', 'homer123.bmp', 'homer21.bmp', 'homer70.bmp', 'bart95.bmp', 'homer66.bmp', 'homer116.bmp', 'homer99.bmp', 'homer17.bmp', 'homer72.bmp', 'bart158.bmp', 'bart119.bmp', 'homer47.bmp', 'bart75.bmp', 'bart166.bmp', 'homer10.bmp', 'bart80.bmp', 'homer50.bmp', 'bart91.bmp', 'bart77.bmp', 'homer98.bmp', 'bart30.bmp', 'bart157.bmp', 'homer113.bmp', 'homer102.bmp', 'bart62.bmp', 'homer105.bmp', 'bart105.bmp', 'bart66.bmp', 'bart130.bmp', 'bart93.bmp', 'bart125.bmp', 'bart102.bmp', 'bart22.bmp', 'bart138.bmp', 'bart26.bmp', 'bart94.bmp', 'bart23.bmp', 'homer122.bmp', 'homer22.bmp', 'bart28.bmp', 'homer80.bmp', 'bart162.bmp', 'bart104.bmp', 'bart128.bmp', 'bart24.bmp', 'homer82.bmp', 'bart141.bmp', 'bart136.bmp', 'bart67.bmp', 'homer2.bmp', 'homer87.bmp', 'bart3.bmp', 'homer35.bmp', 'homer112.bmp', 'bart1.bmp', 'homer73.bmp', 'bart160.bmp', 'bart103.bmp', 'bart38.bmp', 'bart60.bmp', 'bart31.bmp', 'bart73.bmp', 'bart20.bmp', 'homer

In [ ]:
image_size = 128

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),

    # AUGMENTATION
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

In [ ]:
class HomerBartDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # lista de arquivos
        self.image_files = [
            f for f in os.listdir(root_dir)
            if f.lower().endswith(".bmp")
        ]

        # garante ordem estável
        self.image_files.sort()

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.root_dir, img_name)

        # abre imagem
        image = Image.open(img_path).convert("RGB")

        # define o rótulo pela string do nome
        name_lower = img_name.lower()
        if "homer" in name_lower:
            label = 1   # Homer
        elif "bart" in name_lower:
            label = 0   # Bart
        else:
            raise ValueError(f"Nome de arquivo inesperado: {img_name}")

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
dataset = HomerBartDataset(base_path, transform=transform)

print("Total de imagens:", len(dataset))
img0, label0 = dataset[0]
print("Shape da primeira imagem:", img0.shape)
print("Primeiro rótulo:", label0)

NameError: name 'transform' is not defined

In [ ]:
train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size,
                         shuffle=False)

print("Tamanho treino:", len(train_dataset))
print("Tamanho teste:", len(test_dataset))


## Criação da rede

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        # Bloco 1
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)

        # Bloco 2
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)

        # Bloco 3
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)

        self.pool  = nn.MaxPool2d(2, 2)

        # Dropout mais equilibrado
        self.dropout = nn.Dropout(0.4)

        # Camadas fully-connected
        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = torch.flatten(x, 1)

        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)

        return F.log_softmax(x, dim=1)


## Treinamento

### Criando o objeto de treinamento

In [ ]:
def train(model, device, train_loader, optimizer, epoch, criterion, log_interval=10, dry_run=False):
    model.train()
    running_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        # envia dados e rótulos para CPU ou GPU
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()

        # forward
        output = model(data)

        loss = criterion(output, target)

        # backward + atualização dos pesos
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % log_interval == 0:
            print(
                f'Train Epoch: {epoch} '
                f'[{batch_idx * len(data)}/{len(train_loader.dataset)} '
                f'({100. * batch_idx / len(train_loader):.0f}%)]\t'
                f'Loss: {loss.item():.6f}'
            )

        if dry_run:
            break

    avg_loss = running_loss / len(train_loader)
    print(f'Epoch {epoch} finalizada. Loss médio de treino: {avg_loss:.4f}')

In [ ]:
def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)

            output = model(data)

            # soma a loss do batch (média por amostra)
            loss = criterion(output, target)
            test_loss += loss.item() * data.size(0)  # multiplica pelo tamanho do batch

            # pega a classe com maior probabilidade/logit
            pred = output.argmax(dim=1, keepdim=False)
            correct += (pred == target).sum().item()

    # loss média por amostra
    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)

    print(
        f'\nTest set: Average loss: {test_loss:.4f}, '
        f'Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n'
    )

    return test_loss, accuracy


## Avaliação

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)

criterion = torch.nn.NLLLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4   # L2 regularizacao
)

In [ ]:
best_loss = float('inf')
patience = 3
counter = 0

for epoch in range(epochs):
    train(model, device, train_loader, optimizer, epoch, criterion)
    val_loss, val_acc = test(model, device, test_loader, criterion)

    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("\n Early stopping ativado — modelo parou de melhorar.")
        break

### Conclusão



O modelo que desenvolvemos utilizou um dataset personalizado e uma CNN simples, porém eficiente, e ao longo do treinamento nós observamos um aprendizado consistente, com diminuição da loss e aumento da acurácia tanto no treino quanto no teste.

Inicialmente, não identificamos sinais fortes de overfitting, já que a loss de teste também caía ao longo das épocas; ainda assim, percebemos que a loss de treino reduzia mais rapidamente, o que indicava que técnicas adicionais de regularização ou data augmentation poderiam melhorar o desempenho.

Após aplicarmos essas melhorias — incluindo data augmentation, batch normalization, dropout ajustado, weight decay e early stopping — o modelo passou a treinar de forma mais estável e equilibrada, apresentando loss de treino e teste muito mais próximas e mostrando boa capacidade de generalização já nos primeiros epochs.

Embora a acurácia máxima tenha permanecido próxima ao valor anterior (cerca de 80%), o comportamento do modelo tornou-se significativamente mais robusto, menos sensível ao dataset reduzido e menos propenso ao overfitting. Além disso, observamos que o melhor desempenho ocorreu nas primeiras épocas, e o early stopping indicou de forma automática o ponto ideal para interrupção do treinamento, reforçando que continuar por mais épocas não traria ganhos relevantes.

Isso mostra que as técnicas aplicadas tornaram o processo mais consistente e que ajustes mais finos na arquitetura ou nos hiperparâmetros ainda podem elevar a performance futura.